# Singapore at a Glance
This dashboard gives insight into Singapore. The data utilized is retreived from the [SingStat Website](https://www.singstat.gov.sg/).

In [ ]:
%%capture
!pip install pandas
!pip install pyecharts
!pip install duckdb

In [ ]:
import requests
import json
from urllib.parse import urlparse
from urllib.parse import parse_qs
import pandas
import duckdb
import os

In [ ]:
# Prepare Pyecharts if Jupyter Lab notebook
# https://pyecharts.org/#/en-us/notebook?id=jupyter-lab

def get_environment():
    if os.environ.get('FROM_NBCONVERT'):
        return 'nbconvert'
    try:
        shell = get_ipython().__class__.__name__
        if shell == 'ZMQInteractiveShell':
            return 'jupyter-lab'
        else:
            return 'other'
    except:
        return 'other'

if get_environment() == 'jupyter-lab':
    from pyecharts.globals import CurrentConfig, NotebookType
    CurrentConfig.NOTEBOOK_TYPE = NotebookType.JUPYTER_LAB
    from pyecharts.charts import Bar
    Bar().load_javascript()

In [ ]:
import json
from urllib.request import Request,urlopen
def call_api(table, serie_names=[]):
    base_url = "https://tablebuilder.singstat.gov.sg/api/table"
    headers = {'User-Agent': 'Mozilla/5.0', "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,/;q=0.8"}
    def get_serie_number_from_name(serie_name):
        url = f"{base_url}/metadata/{table}"
        request = Request(url,headers=headers)
        data = json.loads(urlopen(request).read())["Data"]
        for row in data['records']['row']:
            if row["rowText"] == serie_name:
                break
        return row["seriesNo"]
    def get_all_serie_numbers():
        url = f"{base_url}/metadata/{table}"
        request = Request(url,headers=headers)
        data = json.loads(urlopen(request).read())["Data"]
        return [row["seriesNo"] for row in data['records']['row']]
    def call_api_serie(serie_no):
        url = f"{base_url}/tabledata/{table}?seriesNoORrowNo={serie_no}"
        request = Request(url,headers=headers)
        data = json.loads(urlopen(request).read())["Data"]
        for row in data['row']:
            if str(row['seriesNo']) == str(serie_no):
                break
        name = row['rowText']
        return [
            {"name": name, "key": i["key"], "value": i["value"]}
            for i in row['columns']
        ]
    if len(serie_names) > 0:
        serie_numbers = []
        for serie_name in serie_names:
            serie_numbers.append(get_serie_number_from_name(serie_name=serie_name))
    else:
        serie_numbers = get_all_serie_numbers()
    result = []
    for serie_no in serie_numbers:
        result = result + call_api_serie(serie_no=serie_no)
    return result

# Singapore Residents By Age Group, Ethnic Group And Gender

## Total Population

In [ ]:
total_population = call_api(table="M810001", serie_names=["Singapore Citizen Population", "Permanent Resident Population", "Non-Resident Population"])
total_population_df = pandas.DataFrame(total_population)
total_population_df = duckdb.sql("""
        select
            case name
                when 'Singapore Citizen Population' then 'Citizens'
                when 'Permanent Resident Population' then 'Permanent Residents'
                when 'Non-Resident Population' then 'Non-Residents'
            end as population_type,
            cast(value as int) as population,
            key as year
        from total_population_df
        where key = (
            select
                max(key)
            from total_population_df
        )
        order by population
    """).df()
total_population_df

In [ ]:
import pyecharts.options as opts
from pyecharts.charts import Pie
from pyecharts.commons.utils import JsCode

x_data = total_population_df['population_type']
y_data = total_population_df['population']
#colors = ["#5470C6", "#91CC75", "#EE6666", "#DDDDDD", "#FCCE10", "#FFFFFF"]  # Define colors for each category

population_pie = (
    Pie()
    .add(
        series_name="",
        data_pair=[list(z) for z in zip(x_data, y_data)],
        radius=["30%", "70%"],
        label_opts=opts.LabelOpts(
            font_size=15,
            formatter=JsCode("function(params){return params.data.name + '\\n' + (params.data.value/10**6).toFixed(2) + 'M'}")
        ),
    )
    .add(
        series_name="Total Population",
        data_pair=[("total", sum(y_data))],
        radius=["0%", "25%"],
        label_opts=opts.LabelOpts(
            is_show=True,
            position="center",
            font_size=30,
            formatter=JsCode("function(params){return (params.data.value/10**6).toFixed(2) + 'M';}")
        ),
    )
    .set_colors(["#1569C7", "#00A36C", "#EB5406", "white"])
    .set_global_opts(
        legend_opts=opts.LegendOpts(is_show=False),  # Hide legend completely
        title_opts=opts.TitleOpts(
            title="Singapore Population Serbe",
            pos_left="center",
            title_textstyle_opts=opts.TextStyleOpts(font_size=30)
        )
    )
    .set_series_opts(
        tooltip_opts=opts.TooltipOpts(
            trigger="item", formatter="{b}: {c} ({d}%)"
        )
    )
)
#population_pie.load_javascript()
population_pie.render_notebook()

## Residents by Ethnic Group

In [ ]:
residents_by_ethnic_group = call_api(
        table="M810011",
        serie_names=["Total Chinese", "Total Malays", "Total Indians", "Other Ethnic Groups (Total)"]
    )

In [ ]:
residents_by_ethnic_group_df = pandas.DataFrame(residents_by_ethnic_group)
residents_by_ethnic_group_df = duckdb.sql("""
        select
            case name
                when 'Total Chinese' then 'Chinese'
                when 'Total Malays' then 'Malays'
                when 'Total Indians' then 'Indians'
                when 'Other Ethnic Groups (Total)' then 'Others'
            end as ethnicity,
            key as year,
            cast(value as int) as population
        from residents_by_ethnic_group_df
    """).df()
residents_by_ethnic_group_df

In [ ]:
import pyecharts.options as opts
from pyecharts.charts import Line

formatter = JsCode("""
    function(value){
        return (value / 1000000).toFixed(0) + 'M'
    }
""")


def get_y_data(ethnic_group):
    query = f"""
        select *
        from residents_by_ethnic_group_df
        where ethnicity = '{ethnic_group}'
        order by year
    """
    result_df = duckdb.sql(query).df()
    return list(result_df['population'])

x_data = sorted([str(k) for k in set(residents_by_ethnic_group_df['year'])])

title = 'Residents by Ethnic Group'


query = """
    select
        ethnicity
    from residents_by_ethnic_group_df
    group by ethnicity
    order by sum(population) desc
"""
ethnicities = duckdb.sql(query).df()['ethnicity']

chart_residents_by_ethnic_group = (
    Line()
    .add_xaxis(xaxis_data=x_data)
)


for ethnicity in ethnicities:
    y_data = get_y_data(ethnicity)
    chart_residents_by_ethnic_group.add_yaxis(
        series_name=ethnicity,
        stack="stack",
        y_axis=y_data,
        label_opts=opts.LabelOpts(is_show=False),
        areastyle_opts=opts.AreaStyleOpts(opacity=0.5),
    )
    
    
chart_residents_by_ethnic_group.set_global_opts(
    title_opts=opts.TitleOpts(
        title=title,
        pos_left="center",
        title_textstyle_opts=opts.TextStyleOpts(font_size=30)
    ),
    tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
    legend_opts=opts.LegendOpts(pos_bottom="20"),
    yaxis_opts=opts.AxisOpts(
        axislabel_opts=opts.LabelOpts(formatter=formatter)
    )
)
chart_residents_by_ethnic_group.render_notebook()

## Population Density

In [ ]:
table = '17560'
base_url = "https://tablebuilder.singstat.gov.sg/api/table"
headers = {'User-Agent': 'Mozilla/5.0', "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,/;q=0.8"}
url = f"{base_url}/tabledata/{table}"
request = Request(url,headers=headers)
data = json.loads(urlopen(request).read())["Data"]
data_list = [
    {
        **{"subzone": row["rowText"]},
            **{f"{k['key']} {i['key']}":i['value']
                for k in row['columns']
                for i in k['columns']
            }
    } 
    for row in data['row']
]
population_df = pandas.DataFrame(data_list)
subzones_population_df = duckdb.sql("""
    with
        base as (
            select
                upper(subzone) as subzone,
                "Total Total" as population
            from population_df
        )
    select
        subzone,
        case
            when population = '-' then 0
            else cast(population as int)
        end as population
    from base
    where lower(subzone) not like '%total'
""").df()
subzones_population_df

In [ ]:
duckdb.sql("""
        install spatial;
        load spatial;
""")
density_df = duckdb.sql("""
    with
        base as (
            select
                subzones_map.name as subzone,
                coalesce(population, 0) / (st_area(geom)/pow(10, 6)) as density
            from st_read('./geojsons/sg_subzones.geojson') as subzones_map
            left outer join subzones_population_df
                on subzones_population_df.subzone = subzones_map.name
        )
    select *
    from base
    where density >= 1000
""").df()
density_df

In [ ]:
import json
from pyecharts.charts import Geo, Map
from pyecharts import options as opts

map_data = density_df.values.tolist()

geojson_path = "./geojsons/sg_subzones.geojson"
with open(geojson_path, "r", encoding="utf-8") as file:
    sg_subzones_geojson = json.loads(file.read())


(
    Map()
        .add_js_funcs("echarts.registerMap('singapore_subzones',{});".format(sg_subzones_geojson))
        .add(
            maptype="singapore_subzones",
            series_name='Density',
            data_pair=map_data,
            label_opts=opts.LabelOpts(is_show=False),
            is_map_symbol_show=False
        )
        .set_global_opts(
            visualmap_opts=opts.VisualMapOpts(max_=30000, min_=1000, pos_left='5%', pos_top='center'),
            legend_opts=opts.LegendOpts(is_show=False),
            title_opts=opts.TitleOpts(
                title="Population Density",
                pos_left="center",
                title_textstyle_opts=opts.TextStyleOpts(font_size=30)
            ),
            tooltip_opts=opts.TooltipOpts(
                trigger="item",
                formatter=JsCode(
                    "function(params) {return params.value ? params.name + ': ' + Math.round(params.value) + ' people per Km²' : params.name;}"
                ) 
            )
        )
        .render_notebook()
)

## Median Gross Monthly Income From Employment

In [ ]:
median_income_data = call_api(table="M920131")
median_income_df = pandas.DataFrame(median_income_data)
median_income_lattest_year_df = duckdb.sql("""
        with
            base as (
                select
                    replace(
                        replace(
                            name,
                            ' - Male',
                            ''
                        ),
                        ' - Female',
                        ''
                    ) as occupation,
                    cast(value as int) as median_income,
                    case
                        when name like '% - Male' then 'Male'
                        else 'Female'
                    end as gender,
                    key as year
                from median_income_df
                where key = (
                    select
                        max(key)
                    from median_income_df
                )
            )
        select
            occupation,
            sum(
                case
                    when gender = 'Male' then median_income
                    else 0
                end
            ) as median_income_male,
            sum(
                case
                    when gender = 'Female' then median_income
                    else 0
                end
            ) as median_income_female,
            year
        from base
        group by
            occupation,
            year
        order by median_income_male
    """).df()
median_income_lattest_year_df

In [ ]:
from pyecharts.charts import Bar, Line, Grid
from pyecharts import options as opts

occupation = [item.split(' (', 1)[0] for item in median_income_lattest_year_df['occupation'].tolist()]

c = (
    Bar()
    .add_xaxis(occupation)
    .add_yaxis("Male", median_income_lattest_year_df['median_income_male'].tolist(), color="#98FB98")
    .add_yaxis("Female", median_income_lattest_year_df['median_income_female'].tolist(), color="#DA70D6")
    .set_series_opts(
        markline_opts=opts.MarkLineOpts(
            precision=0,
            symbol_size=0,
            label_opts=opts.LabelOpts(is_show=False),
            data=[
                opts.MarkLineItem(type_="average", name="Average"),
            ]
        ),
    )
    .set_global_opts(
        xaxis_opts=opts.AxisOpts(splitline_opts=opts.SplitLineOpts(is_show=False)),
        title_opts=opts.TitleOpts(
            title="Median Gross Monthly Salary",
            pos_left="center",
            title_textstyle_opts=opts.TextStyleOpts(font_size=30)
        ),
        legend_opts=opts.LegendOpts(pos_bottom="20")
    )
    .reversal_axis()
)

grid_chart = Grid()
grid_chart.add(
    chart=c,
    grid_opts=opts.GridOpts(
        pos_left="40%",
        pos_right="5%"
    )
)
grid_chart.render_notebook()

## Average Monthly Nominal Earnings Per Employee

In [ ]:
average_monthly_earning_data = call_api(table="M182931")
average_monthly_earning_df = pandas.DataFrame(average_monthly_earning_data)
average_weekly_working_hours_data = call_api(table="M184061", serie_names=["Total"])
average_weekly_working_hours_df = pandas.DataFrame(average_weekly_working_hours_data)

In [ ]:
salary_working_time_df = duckdb.sql("""
        with
            average_monthly_earning as (
                select *
                from (
                    select
                        cast(split_part(key, ' ', 1) as int) as year,
                        cast(replace(split_part(key, ' ', 2), 'Q', '') as int) as quarter,
                        cast(value as int) as average_monthly_salary,
                        row_number() over(
                            partition by year
                            order by quarter desc
                        ) = 1 as is_last_quarter
                    from average_monthly_earning_df
                ) as base
                where is_last_quarter
            ),
            average_weekly_working_hours as (
                select
                    cast(key as int) as year,
                    cast(value as float) as average_weekly_working_hours
                from average_weekly_working_hours_df
            )
        select
            average_monthly_earning.year,
            average_monthly_salary,
            average_weekly_working_hours.average_weekly_working_hours
        from average_monthly_earning
        left outer join average_weekly_working_hours
            on average_monthly_earning.year = average_weekly_working_hours.year
        order by average_monthly_earning.year
    """).df()
salary_working_time_df['average_weekly_working_hours'] = salary_working_time_df['average_weekly_working_hours'].apply(lambda x: round(x, 1))
salary_working_time_df

In [ ]:
x_data = [str(k) for k in salary_working_time_df['year'].tolist()]

bar = (
    Bar()
    .add_xaxis(xaxis_data=x_data)
    .add_yaxis(
        series_name="Average Monthly Salary",
        yaxis_index=0,
        y_axis=salary_working_time_df['average_monthly_salary'].tolist(),
        label_opts=opts.LabelOpts(is_show=False),
    )
    .extend_axis(
        yaxis=opts.AxisOpts(
            name="Average Weekly Working Hours",
            type_="value",
            min_=42,
            max_=48
        )
    )
    .set_global_opts(
        yaxis_opts=opts.AxisOpts(
            name="Average Monthly Salary",
            splitline_opts=opts.SplitLineOpts(is_show=False)
        ),
        title_opts=opts.TitleOpts(
            title="Average Monthly Salary",
            pos_left="center",
            title_textstyle_opts=opts.TextStyleOpts(font_size=30)
        ),
        tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
        legend_opts=opts.LegendOpts(pos_bottom="20")
    )
)

line = (
    Line()
    .add_xaxis(xaxis_data=x_data)
    .add_yaxis(
        series_name="Average Weekly Working Hours",
        yaxis_index=1,
        y_axis=salary_working_time_df['average_weekly_working_hours'].tolist(),
        z=10,
        label_opts=opts.LabelOpts(is_show=False),
        linestyle_opts=opts.LineStyleOpts(width=3)
    )
)

bar.overlap(line).render_notebook()

## Merchandise Trade Balance

In [ ]:
merchandise_imports_data = call_api(table="M451021", serie_names=["Total Merchandise Imports"])
merchandise_imports_df = pandas.DataFrame(merchandise_imports_data)
merchandise_exports_data = call_api(table="M451031", serie_names=["Total Merchandise Exports"])
merchandise_exports_df = pandas.DataFrame(merchandise_exports_data)

In [ ]:
merchandise_trade_balance_df = duckdb.sql("""
        with
            imports as (
                select
                    cast(split_part(key, ' ', 1) as int) as year,
                    sum(cast(value as int)) as imports,
                    count(distinct(key)) as count_months
                from merchandise_imports_df
                group by year
                having count_months = 12
            ),
            exports as (
                select
                    cast(split_part(key, ' ', 1) as int) as year,
                    sum(cast(value as int)) as exports,
                    count(distinct(key)) as count_months
                from merchandise_exports_df
                group by year
                having count_months = 12
            )
        select
            exports.year,
            exports,
            imports,
            exports - imports as trade_balance
        from exports
        inner join imports
            on imports.year = exports.year
        order by exports.year
    """).df()
merchandise_trade_balance_df

In [ ]:
import pyecharts.options as opts
from pyecharts.charts import Bar, Line


formatter = JsCode("""
    function(value){
        return (value / 1000000).toFixed(0) + 'M'
    }
""")

x_data = [str(k) for k in merchandise_trade_balance_df['year']]

line = (
    Line()
    .add_xaxis(x_data)
    .add_yaxis("Exports", merchandise_trade_balance_df['exports'], is_smooth=True, label_opts=opts.LabelOpts(is_show=False))
    .add_yaxis("Imports", merchandise_trade_balance_df['imports'], is_smooth=True, label_opts=opts.LabelOpts(is_show=False))
    .set_global_opts(
        title_opts=opts.TitleOpts(
            title="Merchandise Trade Balance",
            pos_left="center",
            title_textstyle_opts=opts.TextStyleOpts(font_size=30)
        ),
        tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
        legend_opts=opts.LegendOpts(pos_bottom="20"),
        yaxis_opts=opts.AxisOpts(
            axislabel_opts=opts.LabelOpts(formatter=formatter)
        )
    )
)

bar = (
    Bar()
    .add_xaxis(xaxis_data=x_data)
    .add_yaxis(
        series_name="Trade Balance",
        y_axis=merchandise_trade_balance_df['trade_balance'].tolist(),
        label_opts=opts.LabelOpts(is_show=False),
    )
)

line.overlap(bar).render_notebook()